#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt

#### Load the data

In [2]:
from ucimlrepo import fetch_ucirepo

breast_cancer = fetch_ucirepo(id=17)

#### Explore the Data

In [ ]:
X = breast_cancer.data.features
y = breast_cancer.data.targets.squeeze()
X.shape, y.shape, X.info(), y.info(), y.value_counts()

In [4]:
features = breast_cancer.data.features.columns.tolist()

#### Check for duplicates

In [5]:
X.duplicated().sum()

np.int64(0)

In [6]:
y = y.replace({"B": 0, "M": 1})
y = y.astype(int)

#### Split the dataset

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((455, 30), (455,), (114, 30), (114,))

**Preprocessing**: The dataset contains no missing values, and all predictor variables are numeric, so imputation and one-hot encoding are not required. Standardization is applied within the pipelines for Logistic Regression and SVC. Tree-based models (Random Forest, Gradient Boosting, and XGBoost) are evaluated without feature scaling.

####  Define Candidate Model Pipelines

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

GB_pipeline = Pipeline([
    ("clf", GradientBoostingClassifier(random_state=42))
])

XGB_pipeline = Pipeline([
    ("clf", xgb.XGBClassifier(random_state=42))
])

LR_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression())
])

RF_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

SVC_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

### Standardized Cross Validation Evaluation Function

In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
import time
import numpy as np

skfolds = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)


In [10]:
def evaluate_model(model_name, model, x, y, cv):

    y_true_all = []
    y_pred_all = []
    y_score_all = []

    start_time = time.perf_counter()
    fit_count = 0

    for train_index, val_index in cv.split(X=x, y=y):

        clone_clf = clone(model)

        X_train_folds = x.iloc[train_index]
        y_train_folds = y.iloc[train_index]

        X_val_folds = x.iloc[val_index]
        y_val_folds = y.iloc[val_index]

        # ONE fit for this fold
        clone_clf.fit(X_train_folds, y_train_folds)
        fit_count += 1

        # Class predictions
        y_pred = clone_clf.predict(X_val_folds)

        # Continuous scores for AUC
        if hasattr(clone_clf, "predict_proba"):
            y_score = clone_clf.predict_proba(X_val_folds)[:, 1]

        elif hasattr(clone_clf, "decision_function"):
            y_score = clone_clf.decision_function(X_val_folds)

        else:
            raise ValueError(
                f"{model_name} cannot produce scores for AUC"
            )

        # Save this fold's results
        y_true_all.extend(y_val_folds)
        y_pred_all.extend(y_pred)
        y_score_all.extend(y_score)

    runtime = time.perf_counter() - start_time

    # Convert to arrays
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_score_all = np.array(y_score_all)

    # Metrics
    accuracy = accuracy_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)
    f1 = f1_score(y_true_all, y_pred_all)
    auc = roc_auc_score(y_true_all, y_score_all)

    cm = confusion_matrix(
        y_true=y_true_all,
        y_pred=y_pred_all
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "AUC": auc,
        "Runtime_seconds": runtime,
        "Model_evaluations": 1,
        "CV_fits": fit_count,
        "Confusion_matrix": cm
    }

### **AI Agent Workflow**


In [11]:
import os
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")

In [ ]:
from smolagents import (
    ToolCallingAgent, InferenceClientModel,
)  

In [13]:
model = InferenceClientModel(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    provider="deepinfra",
    token=HF_TOKEN,
    temperature=0
)

In [14]:
ML_MODELS = {
    "LR": LR_pipeline,
    "SVC": SVC_pipeline,
    "RF": RF_pipeline,
    "GB": GB_pipeline,
    "XGB": XGB_pipeline
}

In [15]:
ALLOWED_MODELS = list(ML_MODELS.keys())

In [16]:
action_log = []

In [17]:
all_experiment_logs = []
all_action_logs = []
agent_run_summary = []

In [18]:
from smolagents import tool

@tool
def inspect_data() -> str:
    """
    Return predefined summary information about the training dataset.
    """

    summary = {
        "rows": X_train.shape[0],
        "features": X_train.shape[1],
        "missing_values": int(X_train.isnull().sum().sum()),
        "class_distribution": y_train.value_counts().to_dict(),
        "feature_names": X_train.columns.tolist()
    }

    action_log.append({
    "action_number": len(action_log) + 1,
    "action": "inspect_data"
    })
    
    return str(summary)

In [19]:
experiment_log = []
MAX_EVALUATIONS = 5

agent_state = {
    "evaluations": 0,
    "selected_features": list(X_train.columns)
}

In [20]:
@tool
def evaluate_classifier(model_name: str) -> str:
    """
    Evaluate one allowed machine-learning classifier using
    the fixed cross-validation procedure on training data only.

    Args:
        model_name: Classifier to evaluate. Must be one of
            LR, SVC, RF, GB, or XGB.
    """
    if agent_state["evaluations"] >= MAX_EVALUATIONS:
        return "Evaluation budget exhausted."
    
    if model_name not in ML_MODELS:
        return (
            f"Model '{model_name}' is not allowed. "
            "Choose LR, SVC, RF, GB, or XGB."
        )

    selected = agent_state["selected_features"]
    X_selected = X_train[selected]

    agent_state["evaluations"] += 1

    result = evaluate_model(
        model_name,
        ML_MODELS[model_name],
        X_selected,
        y_train,
        skfolds
    )

    experiment_log.append({
        "step": len(experiment_log) + 1,
        "model": model_name,
        "features": selected.copy(),
        "n_features": len(selected),
        "accuracy": result["Accuracy"],
        "precision": result["Precision"],
        "recall": result["Recall"],
        "f1": result["F1"],
        "auc": result["AUC"],
        "runtime": result["Runtime_seconds"],
        "cv_fits": result["CV_fits"]
    })

    return str(result)

In [30]:
@tool
def select_features(features: list[str]) -> str:
    """
    Select training features for subsequent model evaluations.

    Args:
        features: An actual list of feature-name strings chosen from
            the feature names returned by inspect_data.
            Example: ["radius1", "texture1", "area1"].
            Do not pass a schema, dictionary, or type description.
    """

    invalid = [
        f for f in features
        if f not in X_train.columns
    ]

    if invalid:
        return f"Invalid features: {invalid}"

    if len(features) == 0:
        return "At least one feature must be selected."

    agent_state["selected_features"] = features

    action_log.append({
        "action_number": len(action_log) + 1,
        "action": "select_features",
        "features": features.copy(),
        "n_features": len(features)
    })

    return f"Selected {len(features)} features."

In [31]:
def save_run_logs(level, run_id):

    for record in experiment_log:
        saved = record.copy()
        saved["level"] = level
        saved["run_id"] = run_id
        all_experiment_logs.append(saved)

    for record in action_log:
        saved = record.copy()
        saved["level"] = level
        saved["run_id"] = run_id
        all_action_logs.append(saved)

#### Level 1 - Fixed Prediction

In [23]:
@tool
def evaluate_fixed_classifier() -> str:
    """
    Evaluate the predefined classifier using all training features
    and the fixed cross-validation procedure.
    """

    if agent_state["evaluations"] >= 1:
        return "The fixed classifier has already been evaluated."

    agent_state["evaluations"] += 1

    result = evaluate_model(
        "LR",
        LR_pipeline,
        X_train,
        y_train,
        skfolds
    )

    experiment_log.append({
        "step": len(experiment_log) + 1,
        "model": "LR",
        "features": features.copy(),
        "n_features": len(features),
        "accuracy": result["Accuracy"],
        "precision": result["Precision"],
        "recall": result["Recall"],
        "f1": result["F1"],
        "auc": result["AUC"],
        "runtime": result["Runtime_seconds"],
        "cv_fits": result["CV_fits"]
    })

    return str(result)

In [24]:
l1_agent = ToolCallingAgent(
    tools=[evaluate_fixed_classifier],
    model=model,
    max_steps=10
)

In [25]:
def reset_agent_state():
    agent_state["evaluations"] = 0
    agent_state["selected_features"] = list(X_train.columns)

    experiment_log.clear()
    action_log.clear()

In [26]:
reset_agent_state()

start = time.perf_counter()

l1_result = l1_agent.run("""
Your goal is to make use of the tool given to you to complete breast-cancer
classification task.

You may only use the classifier given to you

Use only the available tool and its cross-validation results.
The held-out test set is unavailable.

You have a limited evaluation budget.

When finished, report the classifier that was used and the validation
evidence supporting the decision.
""")

agent1_total_runtime = time.perf_counter() - start

save_run_logs(
    level=1,
    run_id=1
)




╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Your goal is to make use of the tool given to you to complete breast-cancer                                     │
│ classification task.                                                                                            │
│                                                                                                                 │
│ You may only use the classifier given to you                                                                    │
│                                                                                                                 │
│ Use only the available tool and its cross-validation results.                                                   │
│ The held-out test set is unavailable.                                                                           │
│                                                                                                                 │
│ You have a limited evaluation budget.                                                                           │
│                                                                                                                 │
│ When finished, report the classifier that was used and the validation                                           │
│ evidence supporting the decision.                                                                               │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.1-8B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'evaluate_fixed_classifier' with arguments: {}                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {'Model': 'LR', 'Accuracy': 0.9736263736263736, 'Precision': 0.9702380952380952, 'Recall': 
0.9588235294117647, 'F1': 0.9644970414201184, 'AUC': 0.9951909184726522, 'Runtime_seconds': 0.02585860000544926, 
'Model_evaluations': 1, 'CV_fits': 3, 'Confusion_matrix': array(||280,   5],
       |  7, 163]])}

[Step 1: Duration 1.05 seconds| Input tokens: 1,214 | Output tokens: 11]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The classifier used is Logistic Regression (LR) with   │
│ an accuracy of 0.9736, precision of 0.9702, recall of 0.9588, F1 score of 0.9645, and AUC of 0.9952. The        │
│ confusion matrix is [[280, 5], [7, 163]].'}                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The classifier used is Logistic Regression (LR) with an accuracy of 0.9736, precision of 0.9702, 
recall of 0.9588, F1 score of 0.9645, and AUC of 0.9952. The confusion matrix is ||280, 5], |7, 163]].

Final answer: The classifier used is Logistic Regression (LR) with an accuracy of 0.9736, precision of 0.9702, 
recall of 0.9588, F1 score of 0.9645, and AUC of 0.9952. The confusion matrix is [[280, 5], [7, 163]].

[Step 2: Duration 3.09 seconds| Input tokens: 2,619 | Output tokens: 108]

#### Level 2 - Feature Selection

In [32]:
@tool
def evaluate_fixedl2_classifier() -> str:
    """
    Evaluate the fixed Logistic Regression classifier using the
    currently selected training features.
    """

    if agent_state["evaluations"] >= MAX_EVALUATIONS:
        return "Evaluation budget exhausted."

    agent_state["evaluations"] += 1

    selected = agent_state["selected_features"]
    X_selected = X_train[selected]

    result = evaluate_model(
        "LR",
        LR_pipeline,
        X_selected,
        y_train,
        skfolds
    )

    experiment_log.append({
        "step": len(experiment_log) + 1,
        "level": 2,
        "model": "LR",
        "features": selected.copy(),
        "n_features": len(selected),
        "accuracy": result["Accuracy"],
        "precision": result["Precision"],
        "recall": result["Recall"],
        "f1": result["F1"],
        "auc": result["AUC"],
        "runtime": result["Runtime_seconds"],
        "cv_fits": result["CV_fits"]
    })

    return str(result)

In [33]:
l2_agent = ToolCallingAgent(
    tools=[inspect_data, select_features, evaluate_fixedl2_classifier],
    model=model,
    max_steps=10
)

In [34]:
reset_agent_state()

start = time.perf_counter()

l2_result = l2_agent.run("""
Your goal is to perform the breast-cancer classification task.

The classifier is fixed as Logistic Regression.

You may inspect the training-data summary and select which
features to use.

You may not change the classifier.

Use only cross-validation results from the training data.
The held-out test set is unavailable.

When finished, report:
1. the features selected,
2. the fixed classifier used,
3. the validation evidence.
""")

agent2_total_runtime = time.perf_counter() - start

save_run_logs(
    level=2,
    run_id=1
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Your goal is to perform the breast-cancer classification task.                                                  │
│                                                                                                                 │
│ The classifier is fixed as Logistic Regression.                                                                 │
│                                                                                                                 │
│ You may inspect the training-data summary and select which                                                      │
│ features to use.                                                                                                │
│                                                                                                                 │
│ You may not change the classifier.                                                                              │
│                                                                                                                 │
│ Use only cross-validation results from the training data.                                                       │
│ The held-out test set is unavailable.                                                                           │
│                                                                                                                 │
│ When finished, report:                                                                                          │
│ 1. the features selected,                                                                                       │
│ 2. the fixed classifier used,                                                                                   │
│ 3. the validation evidence.                                                                                     │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.1-8B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'inspect_data' with arguments: {}                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {'rows': 455, 'features': 30, 'missing_values': 0, 'class_distribution': {0: 285, 1: 170}, 
'feature_names': |'radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1', 'compactness1', 'concavity1', 
'concave_points1', 'symmetry1', 'fractal_dimension1', 'radius2', 'texture2', 'perimeter2', 'area2', 'smoothness2', 
'compactness2', 'concavity2', 'concave_points2', 'symmetry2', 'fractal_dimension2', 'radius3', 'texture3', 
'perimeter3', 'area3', 'smoothness3', 'compactness3', 'concavity3', 'concave_points3', 'symmetry3', 
'fractal_dimension3']}

[Step 1: Duration 1.90 seconds| Input tokens: 1,491 | Output tokens: 30]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The selected features are: radius1, texture1,          │
│ perimeter1, area1, smoothness1, compactness1, concavity1, concave_points1, symmetry1, fractal_dimension1. The   │
│ fixed classifier used is Logistic Regression (LR). The validation evidence is: Accuracy: 0.9340659340659341,    │
│ Precision: 0.9320987654320988, Recall: 0.888235294117647, F1: 0.9096385542168675, AUC: 0.9814860681114551.'}    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The selected features are: radius1, texture1, perimeter1, area1, smoothness1, compactness1, 
concavity1, concave_points1, symmetry1, fractal_dimension1. The fixed classifier used is Logistic Regression (LR). 
The validation evidence is: Accuracy: 0.9340659340659341, Precision: 0.9320987654320988, Recall: 0.888235294117647,
F1: 0.9096385542168675, AUC: 0.9814860681114551.

Final answer: The selected features are: radius1, texture1, perimeter1, area1, smoothness1, compactness1, 
concavity1, concave_points1, symmetry1, fractal_dimension1. The fixed classifier used is Logistic Regression (LR). 
The validation evidence is: Accuracy: 0.9340659340659341, Precision: 0.9320987654320988, Recall: 0.888235294117647,
F1: 0.9096385542168675, AUC: 0.9814860681114551.

[Step 7: Duration 6.28 seconds| Input tokens: 14,489 | Output tokens: 634]

#### Level 3 Model Selection and Tuning

In [35]:
PARAM_GRID = {
    "LR": {
        "C": [0.1, 1, 10]
    },

    "SVC": {
        "C": [0.1, 1, 10],
        "gamma": ["scale", "auto"]
    },

    "RF": {
        "n_estimators": [100, 300, 500],
        "max_depth": [None, 5, 10]
    },

    "GB": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.05, 0.1]
    },

    "XGB": {
        "n_estimators": [50, 100, 200],
        "max_depth": [3, 6]
    }
}

In [ ]:
from sklearn.base import clone

@tool
def evaluate_configuration(
    model_name: str,
    hyperparameters: dict
) -> str:
    """
    Evaluate an allowed machine-learning model using only permitted
    hyperparameter values.

    Args:
        model_name: Model to evaluate. Must be one of
            LR, SVC, RF, GB, or XGB.
        hyperparameters: Hyperparameters selected for the model.
            Each parameter and value must be allowed by PARAM_GRID.
    """

    # 1. Check evaluation budget
    if agent_state["evaluations"] >= MAX_EVALUATIONS:
        return "Evaluation budget exhausted."

    # 2. Check model is allowed
    if model_name not in ML_MODELS:
        return (
            f"Model '{model_name}' is not allowed. "
            "Choose LR, SVC, RF, GB, or XGB."
        )

    # 3. Get allowed hyperparameters for this model
    allowed_params = PARAM_GRID[model_name]

    # 4. Check each hyperparameter selected by the agent
    for param_name, param_value in hyperparameters.items():

        # Does this parameter exist for this model?
        if param_name not in allowed_params:
            return (
                f"Hyperparameter '{param_name}' is not allowed "
                f"for {model_name}."
            )

        # Is the selected value permitted?
        if param_value not in allowed_params[param_name]:
            return (
                f"Value '{param_value}' is not allowed for "
                f"{model_name}.{param_name}. "
                f"Allowed values: {allowed_params[param_name]}"
            )

    # 5. Get the currently selected features
    selected = agent_state["selected_features"]
    X_selected = X_train[selected]

    # 6. Clone the original pipeline
    configured_model = clone(ML_MODELS[model_name])

    # 7. Apply the selected hyperparameters
    pipeline_params = {
        f"clf__{name}": value
        for name, value in hyperparameters.items()
    }

    configured_model.set_params(**pipeline_params)

    # 8. Count this as one model evaluation
    agent_state["evaluations"] += 1

    # 9. Evaluate using your existing function
    result = evaluate_model(
        model_name,
        configured_model,
        X_selected,
        y_train,
        skfolds
    )


    
    action_log.append({
        "action_number": len(action_log) + 1,
        "action": "evaluate_configuration",
        "model": model_name,
        "hyperparameters": hyperparameters.copy(),git
        "features": agent_state["selected_features"].copy()
    })
    # 10. Log the experiment
    experiment_log.append({
        "step": len(experiment_log) + 1,
        "model": model_name,
        "hyperparameters": hyperparameters.copy(),
        "features": selected.copy(),
        "n_features": len(selected),
        "accuracy": result["Accuracy"],
        "precision": result["Precision"],
        "recall": result["Recall"],
        "f1": result["F1"],
        "auc": result["AUC"],
        "runtime": result["Runtime_seconds"],
        "cv_fits": result["CV_fits"]
    })

    # 11. Return results to the agent
    return str(result)

In [37]:
l3_agent = ToolCallingAgent(
    tools=[
        inspect_data,
        select_features,
        evaluate_configuration
    ],
    model=model,
    max_steps=10
)

In [38]:
reset_agent_state()

start = time.perf_counter()

l3_result = l3_agent.run("""
You are performing Level 3 of a controlled machine-learning experiment.

Your goal is to identify a strong classifier configuration for the
breast-cancer classification task using cross-validation performance
on the training data only.

You may:

1. Inspect summary information about the training data.
2. Select features from the available training features.
3. Choose among ONLY these classifiers:
   - LR
   - SVC
   - RF
   - GB
   - XGB
4. Choose hyperparameter values ONLY from the permitted values
   enforced by the evaluate_configuration tool.
5. Evaluate candidate configurations using the available
   cross-validation evaluation tool.

Constraints:

- You may not access or use the held-out test set.
- You may not use classifiers outside LR, SVC, RF, GB, and XGB.
- You may not use hyperparameter values outside the predefined
  parameter grid.
- You may perform at most the permitted number of model evaluations.
- Base your decisions only on information returned by the available tools.
- Do not invent evaluation results.
- You may inspect the data before model evaluation begins.
- You may select features once before model evaluation begins.
- Once you call evaluate_configuration for the first time,
  the selected features are fixed.
- After model evaluation begins, you must not call inspect_data
  or select_features again.
- You may continue evaluating different permitted models and
  hyperparameter configurations using the fixed feature set.
  

When finished, report:
1. The final classifier selected.
2. The final hyperparameters selected.
3. The features used.
4. The cross-validation results supporting the selection.
5. The configurations you evaluated.
""")

agent3_total_runtime = time.perf_counter() - start

save_run_logs(
    level=3,
    run_id=1
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are performing Level 3 of a controlled machine-learning experiment.                                         │
│                                                                                                                 │
│ Your goal is to identify a strong classifier configuration for the                                              │
│ breast-cancer classification task using cross-validation performance                                            │
│ on the training data only.                                                                                      │
│                                                                                                                 │
│ You may:                                                                                                        │
│                                                                                                                 │
│ 1. Inspect summary information about the training data.                                                         │
│ 2. Select features from the available training features.                                                        │
│ 3. Choose among ONLY these classifiers:                                                                         │
│    - LR                                                                                                         │
│    - SVC                                                                                                        │
│    - RF                                                                                                         │
│    - GB                                                                                                         │
│    - XGB                                                                                                        │
│ 4. Choose hyperparameter values ONLY from the permitted values                                                  │
│    enforced by the evaluate_configuration tool.                                                                 │
│ 5. Evaluate candidate configurations using the available                                                        │
│    cross-validation evaluation tool.                                                                            │
│                                                                                                                 │
│ Constraints:                                                                                                    │
│                                                                                                                 │
│ - You may not access or use the held-out test set.                                                              │
│ - You may not use classifiers outside LR, SVC, RF, GB, and XGB.                                                 │
│ - You may not use hyperparameter values outside the predefined                                                  │
│   parameter grid.                                                                                               │
│ - You may perform at most the permitted number of model evaluations.                                            │
│ - Base your decisions only on information returned by the available tools.                                      │
│ - Do not invent evaluation results.                                                                             │
│ - You may inspect the data before model evaluation begins.                                                      │
│ - You may select features once before model evaluation begins.                                                  │
│ - Once you call evaluate_configuration for the first t

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'inspect_data' with arguments: {}                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {'rows': 455, 'features': 30, 'missing_values': 0, 'class_distribution': {0: 285, 1: 170}, 
'feature_names': |'radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1', 'compactness1', 'concavity1', 
'concave_points1', 'symmetry1', 'fractal_dimension1', 'radius2', 'texture2', 'perimeter2', 'area2', 'smoothness2', 
'compactness2', 'concavity2', 'concave_points2', 'symmetry2', 'fractal_dimension2', 'radius3', 'texture3', 
'perimeter3', 'area3', 'smoothness3', 'compactness3', 'concavity3', 'concave_points3', 'symmetry3', 
'fractal_dimension3']}

[Step 1: Duration 1.84 seconds| Input tokens: 1,872 | Output tokens: 27]

...

[Step 9: Duration 0.65 seconds| Input tokens: 36,540 | Output tokens: 1,235]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 10 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/deepinfra/v1/openai/chat/completions' 
(Request ID: Root=1-6abc036c-1fa553845a5f18fb7a45a94a;24b8795e-a462-41e2-a275-60a0b714fe44)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 10: Duration 0.05 seconds]

AgentGenerationError: Error while generating output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/deepinfra/v1/openai/chat/completions' (Request ID: Root=1-6abc036c-1fa553845a5f18fb7a45a94a;24b8795e-a462-41e2-a275-60a0b714fe44)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

#### Level 4 - Adaptive Workflow

In [ ]:
l4_agent = ToolCallingAgent(
    tools=[
        inspect_data,
        select_features,
        evaluate_configuration
    ],
    model=model,
    max_steps=15
)

In [ ]:
reset_agent_state()

In [ ]:
start = time.perf_counter()

level4_result = l4_agent.run("""
You are performing Level 4 of a controlled machine-learning experiment.

Your goal is to construct a strong classifier configuration for the
breast-cancer classification task using only information available
from the training data.

This is an adaptive workflow. You decide what action to take next
based on the observations returned by previous tool calls.

AVAILABLE ACTIONS

You may:

1. Inspect summary information about the training data.

2. Select or revise the features used for model evaluation.

3. Choose among ONLY these classifiers:
   - LR
   - SVC
   - RF
   - GB
   - XGB

4. Choose hyperparameter values ONLY from the permitted values
   enforced by the evaluate_configuration tool.

5. Evaluate permitted model configurations using cross-validation.

6. After receiving an evaluation result, decide whether to:
   - evaluate another model,
   - try another permitted hyperparameter configuration,
   - revise the selected features,
   - return to an earlier decision,
   - or stop if you believe sufficient evidence has been collected.

OBJECTIVE

Use F1 score as the primary metric for selecting the final
configuration.

Use AUC, recall, precision, and accuracy as secondary supporting
metrics.

CONSTRAINTS

- You may use only information returned by the available tools.
- You may not access or use the held-out test set.
- You may not use classifiers outside LR, SVC, RF, GB, and XGB.
- You may not use hyperparameter values outside the predefined
  parameter grid.
- You may not exceed the model-evaluation budget enforced by the tools.
- Do not invent evaluation results.
- Base subsequent decisions on observations from previous actions.
- You may revisit previous decisions when the observed results justify it.
- Stop when you believe the available evidence is sufficient or when
  the evaluation budget is exhausted.

When finished, report:

1. The final classifier selected.
2. The final hyperparameters selected.
3. The final features selected.
4. The cross-validation metrics for the selected configuration.
5. The configurations evaluated.
6. Any feature-selection changes made during the workflow.
7. A brief explanation of how previous evaluation results affected
   subsequent decisions.
""")

agent4_total_runtime = time.perf_counter() - start

save_run_logs(
    level=4,
    run_id=1
)

In [ ]:
print(level4_result)

In [ ]:
runtime_summary = pd.DataFrame({
    "Level": [1, 2, 3, 4],
    "Agent_Runtime_Seconds": [
        agent1_total_runtime,
        agent2_total_runtime,
        agent3_total_runtime,
        agent4_total_runtime
    ]
})

runtime_summary

In [ ]:
pd.DataFrame(action_log)